In [ ]:
# ==========================================
# 1. Configuración inicial
# ==========================================
import findspark
findspark.init()
 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline
import time
 

spark = SparkSession.builder \
    .appName("MLP_CTR") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "12g") \
    .config("spark.driver.maxResultSize", "12g") \
    .getOrCreate()
 
spark.sparkContext.setLogLevel("WARN")
 
# ==========================================
# 2. Carga del dataset
# ==========================================


# ==========================================
# 2. Carga del dataset + sample temprano
# ==========================================
DATA_PATH = r"C:\Users\camil\Documents\Estudio\DL\Corte1\Dataset\avazu-ctr-prediction\train.gz"
df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

# Tomar muestra ANTES del feature engineering para aliviar memoria
#'0.025'
df = df.sample(fraction=0.0525 , seed=42)
df.cache()
print("Columnas:", df.columns)
print("Número de filas tras sample:", df.count())  

In [ ]:
# ==========================================
# 3. Feature Engineering
# ==========================================
 
# --- Hora del día y franja horaria ---
df = df.withColumn("hour_day", F.col("hour") % 100)
 
df = df.withColumn(
    "franja",
    F.when((F.col("hour_day") >= 0)  & (F.col("hour_day") < 6),  "Madrugada")
     .when((F.col("hour_day") >= 6)  & (F.col("hour_day") < 12), "Mañana")
     .when((F.col("hour_day") >= 12) & (F.col("hour_day") < 18), "Tarde")
     .otherwise("Noche")
)
 
# --- Mapeos categóricos ---

df = df.withColumn("dct_cat", F.col("device_conn_type").cast("string"))
dt_map_expr = (
    F.when(F.col("device_type") == 0, "0")
     .when(F.col("device_type") == 1, "1")
     .otherwise("Otros")
)
df = df.withColumn("dt_cat", dt_map_expr.cast("string"))
 
bp_map_expr = (
    F.when(F.col("banner_pos") == 6, "6")
     .when(F.col("banner_pos") == 7, "7")
     .otherwise("Otros")
)
df = df.withColumn("bp_cat", bp_map_expr.cast("string"))
 
# --- Conteos por grupo (count encoding) ---
# MEJORA DE RENDIMIENTO: usar Window functions es más eficiente
# que múltiples joins de agrupación
count_cols = ['device_ip', 'device_id', 'device_model', 'app_id', 'site_id']
for c in count_cols:
    w = Window.partitionBy(c)
    df = df.withColumn(f"{c}_count", F.count(c).over(w))
 
# --- Features derivadas (ratio encoding) ---
# MEJORA: estos ratios capturan comportamiento relativo entre entidades
df = df.withColumn("app_per_device",
    F.col("app_id_count") / (F.col("device_id_count") + 1))
df = df.withColumn("site_per_device",
    F.col("site_id_count") / (F.col("device_id_count") + 1))

In [ ]:
# ==========================================
# 4. Definir variables
# ==========================================
target_col = "click"
 
# High cardinality → StringIndexer (embedding implícito por índice)
high_card_cols = ['C14', 'C17', 'C19', 'C20', 'C21']
 
# Low cardinality → OneHotEncoder
low_card_cols = ['app_category', 'site_category', 'bp_cat', 'dct_cat',
                 'dt_cat', 'C1', 'C18', 'C15', 'C16', 'franja']
 
# Numéricas (incluyendo count features y ratios)
num_vars = ["hour_day", "device_ip_count", "device_id_count",
                 "app_id_count", "site_id_count", "app_per_device", "site_per_device"
]

In [ ]:
# ==========================================
# 5. Codificación y Pipeline
# ==========================================
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in high_card_cols
]
 
# CORRECCIÓN: low_card_cols primero pasan por StringIndexer antes de OHE
low_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_str_idx", handleInvalid="keep")
    for c in low_card_cols
]
 
ohe = OneHotEncoder(
    inputCols=[f"{c}_str_idx" for c in low_card_cols],
    outputCols=[f"{c}_ohe" for c in low_card_cols],
    handleInvalid="keep"
)
 
assembler_inputs = (
    [f"{c}_idx"  for c in high_card_cols] +
    [f"{c}_ohe"  for c in low_card_cols] +
    num_vars
)
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_unscaled",
    handleInvalid="keep"   # MEJORA: evita errores por nulos en numéricas
)
 
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=False   # CORRECCIÓN: withMean=True falla con vectores dispersos (OHE)
)
 
pipeline_prep = Pipeline(stages=indexers + low_indexers + [ohe, assembler, scaler])
 
df_prepared = pipeline_prep.fit(df).transform(df)
df_prepared = df_prepared.select("features", F.col(target_col).cast("double"))
df_prepared.cache()   # MEJORA: cachear antes del split evita recomputar en train y test

DataFrame[features: vector, click: double]

In [ ]:
# ==========================================
# 6. División estratificada train/test
# MEJORA: randomSplit simple no garantiza distribución de clases.
# La división estratificada mantiene el ratio click/no-click en ambos splits.
# ==========================================
def stratified_split(df, label_col, train_ratio=0.8, seed=42):
    pos = df.filter(F.col(label_col) == 1)
    neg = df.filter(F.col(label_col) == 0)
    pos_train, pos_test = pos.randomSplit([train_ratio, 1 - train_ratio], seed=seed)
    neg_train, neg_test = neg.randomSplit([train_ratio, 1 - train_ratio], seed=seed)
    return pos_train.union(neg_train), pos_test.union(neg_test)
 
train, test = stratified_split(df_prepared, target_col)
 
print(f"Train: {train.count()} filas | Test: {test.count()} filas")
print("Distribución train:", train.groupBy(target_col).count().show())
print("Distribución test:",  test.groupBy(target_col).count().show())

Train: 4499430 filas | Test: 1125146 filas
+-----+-------+
|click|  count|
+-----+-------+
|  1.0| 763933|
|  0.0|3735497|
+-----+-------+

Distribución train: None
+-----+------+
|click| count|
+-----+------+
|  1.0|190745|
|  0.0|934401|
+-----+------+

Distribución test: None


In [ ]:
# ==========================================
# 6b. Undersampling para balancear clases
# ==========================================
count_pos = train.filter(F.col(target_col) == 1.0).count()
count_neg = train.filter(F.col(target_col) == 0.0).count()
total_before = count_pos + count_neg

print("========== ANTES DEL UNDERSAMPLING ==========")
print(f"Total filas train : {total_before}")
print(f"Clase 1 (click)   : {count_pos} ({count_pos/total_before*100:.2f}%)")
print(f"Clase 0 (no click): {count_neg} ({count_neg/total_before*100:.2f}%)")

# Ratio para reducir la clase mayoritaria al tamaño de la minoritaria
ratio = count_pos / count_neg

pos_train = train.filter(F.col(target_col) == 1.0)
neg_train = train.filter(F.col(target_col) == 0.0).sample(fraction=ratio, seed=42)

train_balanced = pos_train.union(neg_train).orderBy(F.rand(seed=42))
train_balanced.cache()

count_pos_b = train_balanced.filter(F.col(target_col) == 1.0).count()
count_neg_b = train_balanced.filter(F.col(target_col) == 0.0).count()
total_after = count_pos_b + count_neg_b

print("\n========== DESPUÉS DEL UNDERSAMPLING ==========")
print(f"Total filas train : {total_after}")
print(f"Clase 1 (click)   : {count_pos_b} ({count_pos_b/total_after*100:.2f}%)")
print(f"Clase 0 (no click): {count_neg_b} ({count_neg_b/total_after*100:.2f}%)")

========== ANTES DEL UNDERSAMPLING ==========
Total filas train : 4499430
Clase 1 (click)   : 763933 (16.98%)
Clase 0 (no click): 3735497 (83.02%)

========== DESPUÉS DEL UNDERSAMPLING ==========
Total filas train : 1526722
Clase 1 (click)   : 763933 (50.04%)
Clase 0 (no click): 762789 (49.96%)


In [ ]:
# ==========================================
# 7. Modelo MLP + CrossValidator
# ==========================================
num_features = len(train_balanced.select("features").first()[0])
print(f"Número de features: {num_features}")
 
mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=target_col,
    seed=42,
    blockSize=128   # MEJORA: blockSize > 1 acelera entrenamiento (mini-batch)
)
 
paramGrid = (
    ParamGridBuilder()
    .addGrid(mlp.layers, [
        [num_features, 10,  2],
        [num_features, 50,  2],
        [num_features, 100, 2],
    ])
    .addGrid(mlp.stepSize, [0.1, 0.01])
    .addGrid(mlp.maxIter,  [100, 200])
    .build()
)
 
evaluator_auc = BinaryClassificationEvaluator(
    labelCol=target_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
 
cv = CrossValidator(
    estimator=mlp,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_auc,
    numFolds=3,
    parallelism=10   
)

Número de features: 118


In [ ]:
# ==========================================
# 8. Entrenamiento
# ==========================================
start_time = time.time()
cv_model = cv.fit(train)
train_time = time.time() - start_time
print(f"Tiempo de entrenamiento: {train_time:.2f} s")

In [ ]:
# ==========================================
# 9. Predicción
# ==========================================
start_pred = time.time()
predictions = cv_model.transform(test)
pred_time = time.time() - start_pred
print(f"Tiempo de predicción: {pred_time:.2f} s")
print(f"Tiempo total (entreno + predicción): {train_time + pred_time:.2f} s")

In [ ]:
# ==========================================
# 10. Evaluación completa
# CORRECCIÓN: el código original solo calculaba F1 y AUC.
# Los requisitos piden también Precisión, Recall y Matriz de Confusión.
# ==========================================
 
# --- AUC ROC ---
auc = evaluator_auc.evaluate(predictions)
 
# --- F1, Precisión, Recall ---
def mc_eval(metric):
    return MulticlassClassificationEvaluator(
        labelCol=target_col, predictionCol="prediction", metricName=metric
    ).evaluate(predictions)
 
f1        = mc_eval("f1")
precision = mc_eval("weightedPrecision")
recall    = mc_eval("weightedRecall")
accuracy  = mc_eval("accuracy")
 
print("\n========== MÉTRICAS ==========")
print(f"AUC-ROC   : {auc:.4f}")
print(f"Accuracy  : {accuracy:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"Precisión : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

In [ ]:
# --- Matriz de confusión ---
print("\n========== MATRIZ DE CONFUSIÓN ==========")
conf_matrix = (
    predictions
    .groupBy(F.col(target_col).alias("Real"),
             F.col("prediction").alias("Predicho"))
    .count()
    .orderBy("Real", "Predicho")
)
conf_matrix.show()

In [ ]:
# --- Mejor modelo encontrado ---
best_model = cv_model.bestModel
print("\n========== MEJOR MODELO ==========")
print(f"Capas      : {best_model.getLayers()}")
print(f"stepSize   : {best_model.getStepSize()}")
print(f"maxIter    : {best_model.getMaxIter()}")
 
spark.stop()